### Camada Bronze

A camada Bronze tem como objetivo armazenar os dados em seu estado original, preservando as características da fonte para posterior análise e tratamento nas etapas subsequentes do pipeline.

In [0]:
%python
# ==============================
# IMPORTAÇÕES
# ==============================

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, sum, when
from pyspark.sql import functions as F
#import seaborn as sns
from matplotlib.ticker import FuncFormatter

In [0]:
%python
# ==============================
# FUNÇÕES ÚTEIS
# ==============================

def importar_dataset():
   urlDados = 'https://raw.githubusercontent.com/marciomalrj/MVPED/main/Vendas.CSV'
   df = pd.read_csv(urlDados, sep=";", encoding="ISO-8859-1")
   return df

def formatar_reais_precisao(x, pos):
    return f'R$ {x:,.0f}'.replace(',', '.')

def formatar_reais(x, pos):
    return f'R$ {x/1e6:.1f}M'

In [0]:
%python
# ==============================
# IMPORTANDO E CARREGANDO OS DADOS
# EXIBINDO PRIMEIRAS LINHAS
# ==============================

df_spark = spark.createDataFrame(importar_dataset())
display(df_spark.limit(5))

In [0]:
%python
# ==============================
# VERIFICANDO COLUNAS VAZIAS
# ==============================
total_linhas = df_spark.count()

for c in df_spark.columns:
    qtd_vazios = df_spark.filter(
        F.col(c).isNull() |
        (F.trim(F.col(c).cast("string")) == "")
    ).count()

    if qtd_vazios == total_linhas:
        print(f"Coluna totalmente vazia: {c}")

Foi identificada uma coluna sem conteúdo útil (Unnamed: 10), contendo apenas valores ausentes. O atributo será removido durante a transformação da camada Bronze para a camada Silver.


In [0]:
%python
# ==============================
# VERIFICANDO LINHAS VAZIAS
# ==============================

colunas_dados = [c for c in df_spark.columns if c != "Unnamed: 10"]

condicao_vazia = " AND ".join([
    f"(TRIM(CAST({c} AS STRING)) = '' OR {c} IS NULL)"
    for c in colunas_dados
])

df_linhas_vazias = df_spark.filter(F.expr(condicao_vazia))

print("Quantidade de linhas vazias:", df_linhas_vazias.count())

display(df_linhas_vazias)

Durante a análise da camada Bronze foram identificadas linhas completamente vazias ao final do arquivo CSV. Como essas linhas não continham informações de negócio, elas serão removidas durante a transformação para a camada Silver, garantindo maior qualidade e consistência dos dados.

In [0]:
%python
# ==============================
# DEMOSTRANDO O ESQUEMA
# ==============================

df_spark.printSchema()

Identificação e Tratamento de Tipos de Dados

Durante a análise inicial da camada Bronze, foi realizada a inspeção da estrutura dos dados por meio do comando printSchema(). Essa verificação permitiu identificar inconsistências entre os tipos de dados carregados e o significado de negócio de alguns atributos.

Observou-se que as colunas DataVenda, PrecoUnitario e CustoUnitario foram importadas como texto (string), embora representem, respectivamente, uma data e valores monetários. A manutenção desses atributos como texto pode comprometer a execução de análises temporais, cálculos financeiros e agregações estatísticas, além de dificultar a aplicação de regras de qualidade dos dados.

Também foi identificado que a coluna Qtd_Vendida foi carregada como número decimal (double). Entretanto, por representar a quantidade de unidades vendidas em cada transação, esse atributo será convertido para um tipo numérico inteiro mais adequado.

Dessa forma, na camada Silver, serão realizadas as conversões necessárias para adequar cada atributo ao seu domínio de negócio. Essa etapa tem como objetivo garantir maior consistência dos dados, melhorar a qualidade das análises e permitir a criação de métricas derivadas, como faturamento, custo total e lucro, de forma confiável.

In [0]:
%python
# ==============================
# VERIFICANDO ESPAÇOS NOS CAMPOS DE TEXTOS
# ==============================

colunas_texto = [
    "Produto",
    "Categoria",
    "Marca",
    "NomeCliente",
    "Pais",
    "Continente"
]

for c in colunas_texto:
    qtd = (
        df_spark
        .filter(F.col(c) != F.trim(F.col(c)))
        .count()
    )

    print(f"{c}: {qtd} registros com espaços extras")

Padronização de valores textuais

Durante a análise de qualidade dos atributos textuais, foi verificada a presença de espaços excedentes no início ou no final dos valores armazenados. A validação foi realizada nas colunas Produto, Categoria, Marca, NomeCliente, Pais e Continente.

Os resultados demonstraram que apenas o atributo Marca apresentou inconsistências desse tipo, com 14.200 registros contendo espaços extras.

Esse tipo de inconsistência pode afetar agrupamentos, filtros e agregações, fazendo com que valores visualmente iguais sejam tratados como categorias distintas. Por esse motivo, na camada Silver, a coluna Marca será ajustada, garantindo maior padronização e consistência dos dados.

In [0]:
%python
# ==============================
# VERIFICANDO REGISTROS DUPLICADOS
# ==============================

total_registros = df_spark.count()
total_distintos = df_spark.dropDuplicates().count()

qtd_duplicados = total_registros - total_distintos

print("Total de registros:", total_registros)
print("Registros distintos:", total_distintos)
print("Registros duplicados:", qtd_duplicados)

Análise de Registros Duplicados

Durante a avaliação da qualidade dos dados, foi realizada uma análise para identificar possíveis registros duplicados no conjunto de dados. Para isso, comparou-se a quantidade total de registros com a quantidade de registros distintos presentes na base.

A análise identificou 203.888 registros totais, dos quais 171.094 são distintos, resultando em 32.794 registros considerados duplicados.

Entretanto, antes de realizar qualquer remoção, é necessário avaliar a natureza dessas ocorrências. Em bases transacionais de vendas, registros aparentemente idênticos podem representar transações legítimas realizadas em momentos distintos e, portanto, não devem ser removidos automaticamente apenas por apresentarem os mesmos valores em seus atributos.

Dessa forma, os registros duplicados serão analisados durante a etapa de transformação para a camada Silver, a fim de determinar se representam duplicidades técnicas decorrentes do processo de coleta e armazenamento ou se correspondem a eventos de negócio válidos. Somente após essa validação será definida a estratégia de tratamento mais adequada.

In [0]:
%python
# ==============================
# REALIZANDO UMA AVALIAÇÃO MAIS PROFUNDA DA DUPLICIDADE
# ==============================
duplicados = (
    df_spark.groupBy(df_spark.columns)
            .count()
            .filter("count > 1")
            .orderBy("count", ascending=False)
)

A análise de duplicidades identificou 32.794 registros repetidos. A investigação demonstrou a existência de registros idênticos em todos os atributos, alguns deles ocorrendo até 48 vezes. Considerando que o conjunto de dados não possui identificador único de transação e que a repetição integral de todos os atributos caracteriza forte indício de duplicidade técnica, esses registros serão removidos durante a transformação para a camada Silver, mantendo apenas uma ocorrência de cada registro.

### Resumo dos problemas encontrados

| Problema         | Evidência                             | Ação na Silver       |
| ---------------- | ------------------------------------- | -------------------- |
| Coluna vazia     | `Unnamed: 10`                         | Remover              |
| Linhas vazias    | Registros sem conteúdo                | Remover              |
| Tipos incorretos | Datas e valores monetários como texto | Converter            |
| Espaços extras   | 14.200 registros em `Marca`           | Aplicar trim         |
| Duplicidades     | 32.794 registros duplicados           | Remover duplicidades |


### Conclusão da Camada Bronze

A análise da camada Bronze permitiu identificar diversos problemas de qualidade presentes nos dados originalmente coletados. Foram encontrados uma coluna sem conteúdo útil, linhas vazias, inconsistências nos tipos de dados, espaços excedentes em atributos textuais e registros duplicados.

Nenhuma alteração foi realizada nesta etapa, uma vez que o objetivo da camada Bronze é preservar os dados em seu estado original. As correções identificadas serão aplicadas durante a transformação para a camada Silver.